Step 4. Simulation

In [ ]:
dataset = TensorDataset(torch.tensor(np.array(adata.X), dtype=torch.float32))
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

input_dim = torch.tensor(np.array(adata.X), dtype=torch.float32).shape[1]
model = TimeSeriesVAE(input_dim, latent_dim=15)

The VAE model is used to learn the distribution (mean μ and variance sigma^2) of the input data. The object function is constructed to include the mean square error reconstruction loss (MSE) and the KL divergence loss and the proportion of the two losses is balanced by the parameter β (default β = 0.5).

The normalized expression matrix of the previous time point (xstart) and the normalized expression matrix of the later time point (xend) were used to train the model, and their respective latent space representations (zstart and zend) were obtained at single-cell level.

If Train_vae is interrupted, it may raise the error: AttributeError: partially initialized module 'torch._dynamo' from '...' has no attribute 'external_utils' (most likely due to a circular import). In this case, restarting the kernel would resolve it.

In [ ]:
torch.manual_seed(42)
trained_model = train_vae(model, dataloader, epochs=50, lr=1e-4) # epochs more than 50 are recomanded

In [ ]:
save_model(trained_model, adata, '/path/to/save/model.pth')

In [ ]:
model = load_model('/path/to/load/model.pth')

In [ ]:
device = next(model.parameters()).device

t1_tensor = torch.tensor(adata[adata.obs.s_e=='start'].X, dtype=torch.float32).to(device)
t2_tensor = torch.tensor(adata[adata.obs.s_e=='end'].X, dtype=torch.float32).to(device)

torch.manual_seed(42)
with torch.no_grad():
        z_t1, _, _ = model.encode(t1_tensor)
        z_t2, _, _ = model.encode(t2_tensor)
        
latent_s = z_t1.cpu().numpy()
latent_e = z_t2.cpu().numpy()
latent_s = pd.DataFrame(data = latent_s, index = adata[adata.obs.s_e=='start'].obs.index)
latent_e = pd.DataFrame(data = latent_e, index = adata[adata.obs.s_e=='end'].obs.index)

Subsequently, for each cell j from zend, the following script calculates Euclidean distances between j and every cell i from zstart, and find the nearest cell i of each cell j. Then, the latent representations of cells of the middle time point were obtained by interpolation in Euclidean space. Finally, the normalized expression matrices of the middle time point xmid were then decoded.

In [ ]:
distances = distance.cdist(latent_s, latent_e, metric='euclidean')
distances = pd.DataFrame(index = adata[adata.obs.s_e=='start'].obs.index, columns = adata[adata.obs.s_e=='end'].obs.index, data = distances)
min_dis = distances.idxmin().to_dict()
latent_mid = pd.DataFrame(latent_e.columns).T
for key,value in min_dis.items():
    cell_end = latent_e.loc[key]
    cell_start = latent_s.loc[value]
    cell_mid = pd.DataFrame((cell_end * 1 + cell_start * 1)/2)
    latent_mid = np.vstack((latent_mid, cell_mid.T))
latent_mid = latent_mid[1:, :]

z_mid = torch.tensor(latent_mid, dtype=torch.float32).to(device)
torch.manual_seed(42)
with torch.no_grad():
    mid_cell = model.decode(z_mid).cpu().numpy()

The following script generate a new AnnData of the middle timepoint (or reset the AnnData object if the normalization in step 5 is incorrectly performed)

In [ ]:
name = new_list = [s[:-1] + "CS8-CytO" if s else s for s in adata[adata.obs.s_e == 'end'].obs.index]

adata_interpolated = sc.AnnData(X=pd.DataFrame(data = mid_cell,
                                              index = adata[adata.obs.s_e == 'end'].obs.index,
                                              columns = adata[adata.obs.s_e == 'end'].var.index),
                               obs = adata[adata.obs.s_e == 'end'].obs)

adata_interpolated.var['features'] = adata_interpolated.var.index

adata_interpolated.obs.time = 'CS8-CytOrigin'
adata_interpolated.obs.s_e = 'mid'
adata_interpolated.obs.index = name